# Topic Sentiment Models

Research code accompanying [Şeker (2026), Expert Systems](https://doi.org/10.1111/exsy.70300).
See the [repository README](../README.md) for dependencies, dataset preparation and citation.

This is an exploratory notebook: select the intended dataset and experiment cells before
running. Local intermediate datasets and manual notebook state are required. Stored
outputs and execution counts have been removed. Model and API identifiers retain the
historical experiment settings; check availability before calling external services.


In [ ]:
from pathlib import Path
import os

# Keep relative data/model paths anchored at the repository root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)


In [ ]:
import pandas as pd
import numpy as np
import pyarrow.ipc as ipc

# Data

##Load

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("data/MABSA_translate_sentiment_topic_all")
df = dataset.to_pandas()

In [ ]:
df.info()

In [ ]:
import pandas as pd
import numpy as np

# ---- Parametreler ----
target_langs = ["es", "fr", "de", "ru", "ja", "zh", "en"]
out_path = "sampled_7langs_balanced_100_each.csv"
RANDOM_SEED = 42

# ---- (Opsiyonel) Aspect dolu kayıt şartı isterseniz, aşağıdakini True yapın ----
REQUIRE_ASPECT = False

# ---- 1) Filtre: diller ----
df7 = df[df["language"].isin(target_langs)].copy()

# ---- 2) Sentiment etiketlerini normalize edelim ----
def norm_sent(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"neg", "negative", "-1"}:
        return "negative"
    if s in {"pos", "positive", "1"}:
        return "positive"
    if s in {"neu", "neutral", "0"}:
        return "neutral"
    return np.nan

df7["sentiment_norm"] = df7["sentiment_polarity"].map(norm_sent)

# Aspect şartı (opsiyonel)
if REQUIRE_ASPECT:
    df7 = df7.dropna(subset=["aspect_term", "aspect_category"])

# Sentiment zorunlu
df7 = df7.dropna(subset=["sentiment_norm"])

# ---- 3) Örnekleme fonksiyonu ----
rng = np.random.default_rng(RANDOM_SEED)

def sample_balanced_100(lang_df: pd.DataFrame, lang: str) -> pd.DataFrame:
    # Sınıf grupları
    groups = {c: lang_df[lang_df["sentiment_norm"] == c] for c in ["negative", "positive", "neutral"]}
    counts = {c: len(g) for c, g in groups.items()}

    # Hedef: çoğunluğa 34, diğerlerine 33
    majority_class = max(counts, key=lambda c: counts[c])
    target_counts = {"negative": 33, "positive": 33, "neutral": 33}
    target_counts[majority_class] = 34

    samples = []
    deficit = 0

    # 1. Tur: min(mevcut, hedef) kadar çek
    for cls in ["negative", "positive", "neutral"]:
        need = target_counts[cls]
        have = counts[cls]
        take = min(need, have)
        if take > 0:
            idx = rng.choice(groups[cls].index.values, size=take, replace=False)
            samples.append(lang_df.loc[idx])
        deficit += (need - take)

    # 2. Tur: açık varsa diğer sınıfların kalanlarından tamamla
    if deficit > 0:
        if samples:
            picked_ids = pd.Index(np.concatenate([s.index.values for s in samples]))
        else:
            picked_ids = pd.Index([])

        leftovers = []
        for cls in ["negative", "positive", "neutral"]:
            remaining = groups[cls].drop(index=picked_ids, errors="ignore")
            if not remaining.empty:
                leftovers.append(remaining)

        if leftovers:
            leftovers_all = pd.concat(leftovers, axis=0)
            take_more = min(deficit, len(leftovers_all))
            if take_more > 0:
                idx_more = rng.choice(leftovers_all.index.values, size=take_more, replace=False)
                samples.append(lang_df.loc[idx_more])
                deficit -= take_more

    # Sonuçları birleştir
    result = pd.concat(samples) if samples else lang_df.head(0)

    # 3. Güvenlik: tam 100'e kırp veya uyar
    if len(result) < 100:
        print(f"[WARN] Dil '{lang}': Yeterli veri yok. Hedef=100, Alınan={len(result)}. Mevcut dağılım={counts}")
        return result
    if len(result) > 100:
        result = result.sample(100, random_state=RANDOM_SEED)

    return result

# ---- 4) Tüm diller için uygula ----
balanced_samples = []
for lang in target_langs:
    lang_df = df7[df7["language"] == lang]
    if lang_df.empty:
        print(f"[WARN] Dil '{lang}' için hiç kayıt yok!")
        continue
    picked = sample_balanced_100(lang_df, lang)
    # Son çıktıyı karıştır
    if not picked.empty:
        picked = picked.sample(frac=1.0, random_state=RANDOM_SEED)
        balanced_samples.append(picked)

final_df = pd.concat(balanced_samples).reset_index(drop=True)

# ---- 5) Kontrol ve kayıt ----
print("Dil dağılımı:\n", final_df["language"].value_counts(dropna=False))
print("\nGenel duygu dağılımı:\n", final_df["sentiment_norm"].value_counts(dropna=False))
print("\nDil x Duygu çapraz tablo (hedef: her dil için ~33/33/34):")
print(pd.crosstab(final_df["language"], final_df["sentiment_norm"]))

final_df.to_csv(out_path, index=False)
print(f"\nSaved -> {out_path}")


In [ ]:
df = final_df

##Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
)

y_true = df["sentiment_polarity"].tolist()
tot_text_len = len(df["text"])

metrics ={}

# Sentiment

In [ ]:
!pip install -q vaderSentiment

In [ ]:
from vaderSentiment import vaderSentiment

vader_model = vaderSentiment.SentimentIntensityAnalyzer()


In [ ]:
def vader_pred_sentiment(texts: pd.Series) -> pd.Series:
  texts_sentiment = texts.apply(lambda text: vader_model.polarity_scores(text)['compound'] if pd.notnull(text) else None)
  return texts_sentiment.apply(lambda x: 'positive' if x >= 0.2 else 'negative' if x <= -0.2 else 'neutral')

## _____

In [ ]:
import time 
# Measure speed
start_time = time.time()
df['vader_sentiment'] = vader_pred_sentiment(df.text)
inference_time = (time.time() - start_time) / tot_text_len

y_pred = df['vader_sentiment'].to_list()


metrics['vader']= {
    "pred_time": inference_time,
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, average="macro"),
    "recall": recall_score(y_true, y_pred, average="macro"),
    "f1": f1_score(y_true, y_pred, average="macro")
}

## HuggingFace

In [ ]:
df.info()

### Tabular

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

import time, requests
import pandas as pd

# Hugging Face API ayarları
HF_TOKEN = os.environ["HF_TOKEN"]

API_URL = "https://api-inference.huggingface.co/models/tabularisai/multilingual-sentiment-analysis"
HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"}

def hf_sentiment_to_df(df, text_col="text", out_col="hf_sentiment", timeout=60, max_retries=3, verbose=True):
    if out_col not in df.columns:
        df[out_col] = pd.Series(index=df.index, dtype=object)

    t0 = time.time()
    filled = 0

    for idx, text in df[text_col].fillna("").astype(str).items():
        retries = 0
        while True:
            r = requests.post(API_URL, headers=HEADERS, json={"inputs": text}, timeout=timeout)

            if r.status_code in (503, 429):
                if retries >= max_retries:
                    if verbose: print(f"⚠️ satır {idx} -> {r.status_code}, None")
                    break
                time.sleep(2 * (retries + 1))
                retries += 1
                continue

            if r.status_code != 200:
                if verbose: print(f"❌ satır {idx} HTTP {r.status_code}")
                break

            out = r.json()

            # Model formatı: [[{label, score}, {...}, ...]]
            if isinstance(out, list) and len(out) > 0 and isinstance(out[0], list):
                candidates = out[0]
                best = max(candidates, key=lambda x: x.get("score", 0.0))
                df.at[idx, out_col] = str(best.get("label", "")).lower()
                filled += 1
            else:
                if verbose: print(f"❓ satır {idx} beklenmeyen format: {out}")
            break

    per_sample = (time.time() - t0) / max(1, len(df))
    if verbose:
        print(f"✅ doldurulan: {filled}/{len(df)} | per-sample: {per_sample:.6f}s")
    return per_sample

# --- kullanım ---
per_sample = hf_sentiment_to_df(df, text_col="text", out_col="hf_sentiment", verbose=True)
print(df["hf_sentiment"].value_counts(dropna=False))


In [ ]:
# hf_sentiment -> hf_sentiment_3cls dönüşümü
mapping = {
    "very positive": "pozitive",
    "positive": "pozitive",
    "very negative": "negative",
    "negative": "negative",
    "neutral": "neutral"
}

df["hf_sentiment_3cls"] = df["hf_sentiment"].map(mapping)

print(df["hf_sentiment_3cls"].value_counts())


In [ ]:
df.head()

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

import requests
import pandas as pd

# Hugging Face token is read from the local environment.
HF_TOKEN = os.environ["HF_TOKEN"]
headers = {"Authorization": f"Bearer {HF_TOKEN}"}

# Hugging Face modeli (3 sınıf: negatif, nötr, pozitif)
API_URL = "https://api-inference.huggingface.co/models/cardiffnlp/twitter-roberta-base-sentiment-latest"

def query_hf(text):
    payload = {"inputs": text}
    response = requests.post(API_URL, headers=headers, json=payload)
    response.raise_for_status()
    return response.json()

def hf_sentiment(df, text_col, out_col):
    df[out_col] = None
    for idx, text in df[text_col].fillna("").astype(str).items():
        try:
            result = query_hf(text)
            if isinstance(result, list) and len(result) > 0 and isinstance(result[0], list):
                scores = result[0]
                # en yüksek skorlu etiketi al
                label = max(scores, key=lambda x: x['score'])['label'].lower()
                # negatif / pozitif / nötr formatına çevir
                if "neg" in label:
                    label = "negative"
                elif "pos" in label:
                    label = "positive"
                else:
                    label = "neutral"
                df.at[idx, out_col] = label
        except Exception as e:
            print(f"❌ {idx}: {e}")
    return df

# --- Örnek kullanım ---
data = {
    "text": [
        "I love this product!",
        "It is okay, nothing special.",
        "I hate waiting in long lines."
    ]
}
dfsfsd = pd.DataFrame(data)

dfsfsd = hf_sentiment(dfsfsd, text_col="text", out_col="hf2_sentiment")
print(dfsfsd)
print(dfsfsd["hf2_sentiment"].value_counts())


In [ ]:
df.head()

### ____

# Sentiment Metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

metrics_df = pd.DataFrame(metrics).T.reset_index().rename(columns={'index': 'Model'})
metrics_df['pred_time_ms'] = metrics_df['pred_time'] * 1000

# ===== Plot Metric Comparison =====
plt.figure(figsize=(8,5))
sns.lineplot(data=metrics_df, x='variable', y='value', hue='Model', marker='o')
plt.title("Model Performance Trend Across Metrics")
plt.ylim(0, 1)
plt.show()

heatmap_data = metrics_df.set_index('Model')[['accuracy','precision','recall','f1']]
plt.figure(figsize=(6,4))
sns.heatmap(heatmap_data, annot=True, cmap='YlGnBu', fmt=".3f")
plt.title("Model Metrics Heatmap")
plt.show()

# ===== Plot Prediction Time Comparison =====
metrics_df['time_per_1000'] = metrics_df['pred_time_ms'] * 1000
slowest_time = metrics_df['pred_time_ms'].max()
metrics_df['relative_speed'] = slowest_time / metrics_df['pred_time_ms']

fig, ax1 = plt.subplots(figsize=(7,5))
sns.barplot(data=metrics_df, x='Model', y='pred_time_ms', ax=ax1, palette='viridis', hue='Model')

max_height = metrics_df['pred_time_ms'].max()
ax1.set_ylim(0.0001, max_height * 3)


for i, row in metrics_df.iterrows():
    ax1.text(i, row['pred_time_ms'] + 0.0005,
             f"{row['pred_time_ms']:.4f} ms\n({row['relative_speed']:.1f}x faster)",
             ha='center', fontsize=9)

ax1.set_yscale('log')
ax1.set_ylabel("Prediction Time per Sample (ms, log scale)")
ax1.set_title("Detailed Prediction Time Comparison")
plt.show()
# ===== Confusion Matrices =====
for model in ['genai_sentiment_score', 'vader_sentiment', 'tabular_sentiment', 'senti_sentiment']:
    cm = confusion_matrix(y_true, df[model], labels=["positive","neutral","negative"])
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["positive","neutral","negative"],
                yticklabels=["positive","neutral","negative"])
    plt.title(f"Confusion Matrix - {model}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

# Topic

## Text Preprocess

In [ ]:
import nltk

from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
from nltk.corpus import stopwords
nltk.download('stopwords')

import re

import requests as req
url = "https://gist.githubusercontent.com/ZohebAbai/513218c3468130eacff6481f424e4e64/raw/b70776f341a148293ff277afa0d0302c8c38f7e2/gist_stopwords.txt"
stop_words = req.get(url).text

In [ ]:
lemmatizer = WordNetLemmatizer()

token_reviews = []
def clear_Text(rev):

  if pd.isnull(rev):
    return None

  rev = re.sub('[^a-zA-Z]', ' ', rev)
  rev = rev.lower()
  rev = rev.split()
  rev = [token for token in rev if token not in stop_words]
  rev = [lemmatizer.lemmatize(token) for token in rev]
  rev = [token for token in rev if len(token)>3]      #     OPTIONAL

  token_reviews.append(rev)

  return ' '.join(rev)


In [ ]:
df.translated_text = df.apply(lambda row: row.translated_text if pd.notnull(row.translated_text) else row.text  ,axis=1)

In [ ]:
df['cleaned_text'] = df.translated_text.apply(clear_Text)

In [ ]:
!pip install -q gensim

In [ ]:
import gensim
import gensim.corpora as corpora

In [ ]:
# Mapping from word IDs to words
id2word = corpora.Dictionary(token_reviews)

corpus = []
for token_doc in token_reviews:
    corpus.append(id2word.doc2bow(token_doc))

## LDA

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [ ]:
nltk_stop_words = list(stopwords.words('english'))

vectorizer = CountVectorizer(max_df=0.95, min_df=1, stop_words=nltk_stop_words)
doc_term_matrix = vectorizer.fit_transform(df.cleaned_text)

In [ ]:
lda_model = LatentDirichletAllocation(n_components=3, random_state=42)
lda_topics = lda_model.fit_transform(doc_term_matrix)

In [ ]:
n_top_words=7
feature_names = vectorizer.get_feature_names_out()
topic_keywords = []
for topic in lda_model.components_:
    top_keywords = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
    topic_keywords.append(", ".join(top_keywords))

In [ ]:
df['lda_topic'] = np.argmax(lda_topics, axis=1)

df['lda_topic'] = df['lda_topic'].apply(lambda x: topic_keywords[x])

##NMF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=1, ngram_range=(1, 2))
tfidf = tfidf_vectorizer.fit_transform(df.cleaned_text)

In [ ]:
nmf_model = NMF(n_components=3, random_state=42)
nmf_topics = nmf_model.fit_transform(tfidf)

In [ ]:
topic_labels = []
top_n=10
feature_names = tfidf_vectorizer.get_feature_names_out()
for idx, topic in enumerate(nmf_model.components_):
    topic_labels.append(", ".join([feature_names[i] for i in topic.argsort()[:-top_n - 1:-1]]))

In [ ]:
df['nmf_topic'] = np.argmax(nmf_topics, axis=1)

df['nmf_topic'] = df['nmf_topic'].apply(lambda x: topic_labels[x])

## BERTopic

In [ ]:
!pip install -q bertopic
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

cluster_model = HDBSCAN(min_cluster_size = 15,
                        metric = 'euclidean',
                        cluster_selection_method = 'eom',
                        prediction_data = True)

# BERTopic model
bert_model = BERTopic(
                      language = "english",
                      calculate_probabilities=False,
                      verbose=True,
                      embedding_model = embedding_model,
                      hdbscan_model = cluster_model)

# Fit the model on a corpus
bert_topics, probs = bert_model.fit_transform(df.cleaned_text.to_list())

In [ ]:
# Her yorum için anahtar kelime içeren konuyu al
topic_labels = bert_model.get_topic_info()
topic_keywords = bert_model.get_topics()

# Yorumlara konuların anahtar kelimelerini ata
df["bert_topics"] = bert_topics
df["bert_topics"] = df["bert_topics"].apply(
    lambda top_int:
      ", ".join([topic_tuple[0] for topic_tuple in bert_model.get_topic(top_int)])
    if top_int != -1
      else None
)


#### Görselleştirme

In [ ]:
bert_topic_words = {}
for topic, words in enumerate(bert_model.get_topic_info()['Representation']):
  bert_topic_words[ topic-1 ] = ' '.join(words)
bert_topic_words

In [ ]:
bert_model.visualize_barchart()

## Top2Vec

In [ ]:
!pip install top2vec
!pip install top2vec[sentence_encoders]

from top2vec import Top2Vec

In [ ]:
reviews = df.cleaned_text.to_list()

t2v_model = Top2Vec(
  documents= reviews,
  embedding_model = 'universal-sentence-encoder-large',
  speed = 'deep-learn',
  workers = 4,
  umap_args = None,
  hdbscan_args = None,
  min_count=10 # T2V bir kelimeyi değerlendirmesi için min bu sayı kadar tekrar edilmiş olması gerek default 50 (küçük veri parçasında yaptığım için 10'a düşürdüm)
  )

In [ ]:
topic_nums, _, topics_words, _ = model.get_documents_topics( list(range(len(reviews))))

In [ ]:
label_mapping = {
    topic_id: ", ".join(topics_words[idx])
    for idx, topic_id in enumerate(topic_nums)
}

In [ ]:
df['t2v_topic'] = [label_mapping[t] for t in topic_nums]


# _____